# Stock Analysis
Braulio Solorio
</br>
**Hypothesis:** Can stocks be predicted?
</br>
**Data Source:** https://www.kaggle.com/datasets/ibrahimshahrukh/top-10-s-and-p-500-stocks-2010-2026-analysis

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Custom color palette - vibrant and modern
custom_palette = ['#9500ff', '#ff0059', '#ff8c00', '#b4e600', '#0fffdb']

# Set elegant minimalist theme with clean sans-serif fonts
sns.set_theme(
    context='paper',
    style='whitegrid',
    palette=custom_palette,
    rc={
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
        'axes.labelcolor': '#2962ff',
        'axes.edgecolor': '#e0e0e0',
        'axes.linewidth': 0.8,
        'grid.color': '#f5f5f5',
        'grid.linewidth': 0.5,
        'xtick.color': '#666666',
        'ytick.color': '#666666',
        'text.color': '#333333',
        'axes.titlesize': 14,
        'axes.titleweight': 'bold',
        'axes.labelsize': 11,
        'axes.labelweight': 'bold',
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'font.size': 10
    }
)

In [0]:
%sql --quiet
select
  *
from
  stockanalysis.stocks;

In [0]:
stocks_df = _sqldf.toPandas()
stocks_df.head(5)

In [0]:
stocks_df.shape

## 1. Exploratory Data Analysis


In [0]:
%sql
select 
    ticker, count(ticker) as count, min(date) as fisrt_date, min(adj_close) as min_price, max(adj_close) as max_price 
from
    stockanalysis.stocks
group by ticker

In [0]:
df = _sqldf.toPandas()
sns.barplot(data=df, x='ticker', y='count', color='#2962ff')
for i, row in df.iterrows():
    plt.text(i, row['count'] - (0.05 * row['count']), f"{row['count']}", color='white', ha='center', va='top', fontsize=10, weight='bold')
max_idx = df['count'].idxmax()
max_val = df.loc[max_idx, 'count']
plt.axhline(max_val, color='#ff8c00', linestyle='--', linewidth=1.5, alpha=0.8)
plt.text(len(df)-0.5, max_val, f'{max_val}', color='#ff8c00', fontsize=11, va='bottom', weight='bold')
plt.xlabel('Ticker', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
sns.despine(left=True, bottom=True)

Some important notes, the IPO (Initial Public Offer) from Meta (META) and Tesla (TSLA) where after 2010, 
resulting in less registers

In the result above, it is visible the top 8 companies in the stock market (GOOG and GOOGL are class C and A respectively from alphabet).
As shown in the lineplot below the tickers **GOOG** and **GOOGL** are practically the same. For this analysis GOOG is dropped

In [0]:
goog_vs_googlb = stocks_df[stocks_df['Ticker'].isin(['GOOG', 'GOOGL'])]

plt.figure(figsize=(14, 7))
sns.lineplot(x='Date', y='Close', hue='Ticker', data=goog_vs_googlb, linewidth=2)
plt.title('GOOG vs GOOGL', fontweight='bold', pad=20)
plt.xlabel('Date', fontweight='bold')
plt.ylabel('Close Price ($)', fontweight='bold')
plt.legend(title='Ticker', title_fontsize=11, fontsize=10, frameon=True, fancybox=False, edgecolor='#e0e0e0')
sns.despine()
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT 
    * 
FROM 
    stockanalysis.stocks 
WHERE
    ticker <> "GOOG"

In [0]:
stocks_df = _sqldf.toPandas()
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
stocks_df.head(5)

In [0]:
stocks_df.shape

In [0]:
def plot_ticker_highlight(stocks_df, highlight_ticker, highlight_color='#2962ff', other_color='#aaa8'):
    plt.figure(figsize=(14, 7))
    # Plot all tickers except the highlighted one
    other_df = stocks_df[stocks_df['Ticker'] != highlight_ticker]
    sns.lineplot(x='Date', y='Close', hue='Ticker', data=other_df, linewidth=2, palette=[other_color]*other_df['Ticker'].nunique(), legend=False)
    # Plot the highlighted ticker
    highlight_df = stocks_df[stocks_df['Ticker'] == highlight_ticker]
    sns.lineplot(x='Date', y='Close', data=highlight_df, linewidth=3, color=highlight_color, label=highlight_ticker)
    plt.xlabel('Date', fontweight='bold')
    plt.ylabel('Close Price ($)', fontweight='bold')
    plt.legend(title='Ticker', title_fontsize=11, fontsize=10, frameon=True, fancybox=False, edgecolor='#e0e0e0')
    sns.despine()
    plt.tight_layout()
    plt.show()

In [0]:
plot_ticker_highlight(stocks_df, 'AAPL')


In [0]:
stocks_df["Valuation"] = stocks_df["Adj_Close"] * stocks_df["Volume"]
stocks_df.head(5)
plt.figure(figsize=(14, 7))
sns.lineplot(x='Date', y='Valuation', hue='Ticker', data=stocks_df, linewidth=2)
plt.title('Stock Market Capitalization 2010 - 2026', fontweight='bold', pad=20)
plt.xlabel('Date', fontweight='bold')
plt.ylabel('Valuation ($)', fontweight='bold')
plt.legend(title='Ticker', title_fontsize=11, fontsize=10, frameon=True, fancybox=False, edgecolor='#e0e0e0')
sns.despine()
plt.tight_layout()
plt.show()  

In [0]:
from matplotlib.patches import ConnectionPatch

# Gradient palette for main color (blue) + grey for "Others"
main_color = '#2962ff'
gradient_colors = sns.light_palette(main_color, n_colors=len(df), reverse=True).as_hex()
colors = gradient_colors + ['#bdbdbd3a']  # grey for "Others"

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    df_complete['Percentage'],
    labels=df_complete['Company'],
    autopct=lambda pct: f"{pct:.2f}%",
    startangle=140,
    textprops={'fontsize': 11, 'color': '#000', 'weight': 'light'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1},
    colors=colors,
    pctdistance=0.85  # Move percentage closer to border
)

# Tilt percentage labels to corresponding angle
for i, autotext in enumerate(autotexts):
    angle = (wedges[i].theta2 + wedges[i].theta1) / 2
    if i == len(autotexts) - 1:
        angle -= 180
    autotext.set_rotation(angle + 180) 
    autotext.set_va('center')
    autotext.set_ha('center')
    autotext.set_color('#000')

plt.title("VOO Portfolio")
plt.tight_layout()
plt.show()

## Using Recurrent Neural Networks

In [0]:
!pip install tensorflow keras protobuf==3.20.*

In [0]:
import math
from keras.models import Sequential
from keras.layers import Dense
from keras import Input  # make sure this is imported

In [0]:
def create_dataset(dataset, look_back=12):
  dataX, dataY = [], []
  for i in range(len(dataset)-look_back-1):
    a = dataset[i:(i+look_back), 0]
    dataX.append(a)
    dataY.append(dataset[i + look_back, 0])
  return np.array(dataX), np.array(dataY)

In [0]:
%skip
from keras.layers import LSTM

LOOK_BACK = 60
TRAIN_SIZE = 0.8
TEST_SIZE = 1 - TRAIN_SIZE

def predict_stock(ticker):
    # --- Data prep ---
    ticker_stocks = stocks_df[stocks_df['Ticker'] == ticker].copy()
    ticker_stocks['Date'] = pd.to_datetime(ticker_stocks['Date'])
    ticker_stocks = ticker_stocks.sort_values(by='Date').reset_index(drop=True)

    dataset = ticker_stocks['Adj_Close'].values.reshape(-1, 1)

    # --- Sanity check BEFORE splitting ---
    min_rows = int(np.ceil((LOOK_BACK + 2) / (1 - TEST_SIZE)))
    assert len(dataset) >= min_rows, (
        f"Only {len(dataset)} rows for {ticker}, need at least {min_rows} "
        f"for LOOK_BACK={LOOK_BACK} and TEST_SIZE={TEST_SIZE}"
    )

    # --- Split ---
    train_size = int(len(dataset) * TRAIN_SIZE)
    train = dataset[:train_size]
    test  = dataset[train_size:]

    # --- Windows ---
    trainX, trainY = create_dataset(train, LOOK_BACK)
    testX,  testY  = create_dataset(test,  LOOK_BACK)

    # --- Shape guard for LSTM ---
    trainX = trainX.reshape(-1, LOOK_BACK, 1)
    testX  = testX.reshape(-1,  LOOK_BACK, 1)

    print(f"[{ticker}] dataset={len(dataset)}, train={len(train)}, test={len(test)}")
    print(f"[{ticker}] trainX={trainX.shape}, testX={testX.shape}")

    # --- LSTM Model ---
    model = Sequential([
        Input(shape=(LOOK_BACK, 1)),
        LSTM(32, return_sequences=False),
        Dense(8, activation='relu'),
        Dense(4, activation='relu'),
        Dense(1)
    ])
    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(trainX, trainY, epochs=100, batch_size=2, verbose=2)

    # --- Predict ---
    train_pred = model.predict(trainX)
    test_pred  = model.predict(testX)

    trainScore = model.evaluate(trainX, trainY, verbose=0)
    print('Train Score: %.2f MSE (%.2f RMSE)' % (trainScore, math.sqrt(trainScore)))
    testScore = model.evaluate(testX, testY, verbose=0)
    print('Test Score: %.2f MSE (%.2f RMSE)' % (testScore, math.sqrt(testScore))

    # --- Align predictions with original index ---
    padding = np.full(LOOK_BACK + 1, np.nan)
    ticker_stocks['Train_Pred'] = np.concatenate([padding, train_pred.reshape(-1), np.full(len(ticker_stocks) - len(padding) - len(train_pred), np.nan)])
    ticker_stocks['Test_Pred']  = np.concatenate([np.full(train_size, np.nan), padding, test_pred.reshape(-1), np.full(len(ticker_stocks) - train_size - len(padding) - len(test_pred), np.nan)])

    # --- Forecast next 2 months ---
    last_dates = ticker_stocks['Date'].iloc[-LOOK_BACK:]
    last_values = ticker_stocks['Adj_Close'].iloc[-LOOK_BACK:].values
    future_preds = []
    future_dates = []
    last_date = ticker_stocks['Date'].iloc[-1]
    for i in range(60):  # 2 months ~ 60 days
        input_seq = last_values.reshape(1, LOOK_BACK, 1)
        pred = model.predict(input_seq)[0][0]
        future_preds.append(pred)
        future_dates.append(last_date + pd.Timedelta(days=i+1))
        last_values = np.append(last_values[1:], pred)

    # --- Prepare forecast DataFrame ---
    forecast_df = pd.DataFrame({
        'Date': future_dates,
        'Forecast': future_preds
    })

    # --- Plot ---
    plt.figure(figsize=(14, 7))
    sns.lineplot(x='Date', y='Adj_Close', data=ticker_stocks, label='Actual', color='#333333', linewidth=2.5)
    sns.lineplot(x='Date', y='Train_Pred', data=ticker_stocks, label='Train', color='#2962ff', linewidth=2, alpha=0.8)
    sns.lineplot(x='Date', y='Test_Pred', data=ticker_stocks, label='Test', color='#ff8c00', linewidth=2, alpha=0.8)
    sns.lineplot(x='Date', y='Forecast', data=forecast_df, label='Next 2 Months', color='#b4e600', linewidth=2.5, linestyle='--')

    # --- Add label for last predicted value and date ---
    last_pred_value = future_preds[-1]
    last_pred_date = future_dates[-1]
    plt.annotate(f"${last_pred_value:.2f}\n{last_pred_date.date()}",
                 xy=(last_pred_date, last_pred_value),
                 xytext=(last_pred_date, last_pred_value + 10),
                 arrowprops=dict(arrowstyle="->", color='#b4e600', lw=1.5),
                 color='#b4e600', fontsize=10, weight='bold',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#b4e600', alpha=0.9))

    plt.title(f'{ticker} Stock Prediction & 2-Month Forecast', fontweight='bold', pad=20, fontsize=14)
    plt.xlabel('Date', fontweight='bold')
    plt.ylabel('Adjusted Close Price ($)', fontweight='bold')
    plt.legend(title='', fontsize=10, frameon=True, fancybox=False, edgecolor='#e0e0e0', loc='best')
    sns.despine()
    plt.tight_layout()
    plt.show()

In [0]:
%skip
predict_stock("META")

In [0]:
%skip
def stocastic_predict(ticker):
    # --- Data prep ---
    ticker_stocks = stocks_df[stocks_df['Ticker'] == ticker].copy()
    ticker_stocks['Date'] = pd.to_datetime(ticker_stocks['Date'])
    ticker_stocks = ticker_stocks.sort_values(by='Date').reset_index(drop=True)

    dataset = ticker_stocks['Adj_Close'].values.reshape(-1, 1)
    print(dataset)

In [0]:
%skip
stocastic_predict("META")